# Step 6 — LambdaRank re-ranker and the final comparison

Combines the hand-crafted features with the cross-fitted tower score in a LightGBM LambdaRank model and compares every model on the test users.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [1]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

config configs/default.yaml | scope full | raw data /home/ubuntu/bt4222-two-tower-recsys/data/raw | outputs /home/ubuntu/bt4222-two-tower-recsys/outputs/full


## 6.1 Re-ranker (`scripts/05_train_ranker.py`, `src/jds/ranker/`)

LightGBM `lambdarank`, label gain [0, 1, 3, 7] (= 2^label − 1), trained on training users' slates, early-stopped on
validation users. Inputs: the step-2 features + tower score, its within-slate percentile and gap to the slate's best.
Needs ~15 GB RAM at full scale: on a standard Colab runtime add `"--set", "ranker.train_user_frac=0.5"`.

### Why these inputs and this setup

* **Out-of-fold inputs.** Training users' tower scores and CTR features come from models and tables that never saw their slates. A tower score fitted on the same slate would look far better than it does on new users, and the ranker would trust it too much.
* **Slate-relative tower features.** Cosine scales differ between users, so `tower_score_pct` (rank within the slate) and `tower_score_gap` (distance to the slate's best) give comparisons the ranker can apply to any slate.
* **Positives-only slates in training and validation.** A slate with no positive has no pair to order, so it gives LambdaRank nothing to learn from and leaves NDCG undefined.
* **Regularisation.** Parameters in `configs/default.yaml` limit overfitting to training slates: `num_leaves: 31`, `max_depth: 7` and `min_data_in_leaf: 1000` keep trees simple; `feature_fraction: 0.5` and `bagging_fraction: 0.7` subsample columns and rows; `lambda_l2: 5.0` and `path_smooth: 10.0` shrink leaf values. Up to 3,000 trees are allowed at learning rate 0.03, and early stopping sets the actual number.

### Key functions (`src/jds/ranker/`)

* **`dataset.tower_features`** adds the two slate-relative tower features.
* **`dataset.load`** joins features and tower scores, sorts by user so each slate is one contiguous LightGBM group, and stops with an error if any row has no tower score (a sign the two inputs came from different scopes).
* **`train.fit_frames`** builds the LightGBM datasets with one query group per user and trains LambdaRank. Boosting stops when validation NDCG has not improved for 100 rounds. `train.fit` loads the rows and calls it; the ablations reuse it with fewer features or users.
* **`train.train`** saves the model at its best iteration with the feature order and importances.
* **`infer.predict`** scores every row of a split. **`infer.run`** evaluates train, validation and test users. **`infer.summary`** gathers every model's test metrics into one table.

In [2]:
run.stage("05_train_ranker.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/05_train_ranker.py --config configs/default.yaml


06:16:08 ranker | ranker exists (use --force to retrain)


In [3]:
run.stage("06_predict.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/06_predict.py --config configs/default.yaml


06:17:22 ranker | 


shape: (3, 9)


┌──────────────────────────────┬────────┬────────┬────────┬───┬────────┬────────┬────────┬─────────┐


│ model                        ┆ mrr    ┆ p1     ┆ ndcg10 ┆ … ┆ hr10   ┆ r10    ┆ r50    ┆ n_users │


...

│ Prior team Hybrid NCF+LSTM (fixed) [test]             ┆ 0.1708 ┆ 0.0734 ┆ 0.1644 ┆ 0.1619  ┆ 0.3706 ┆ 0.2851 ┆ 0.7288 ┆ 17174   │


│ Prior team Hybrid NCF+LSTM (published) [test]         ┆ 0.135  ┆ 0.0597 ┆ 0.122  ┆ 0.1195  ┆ 0.2812 ┆ 0.2055 ┆ 0.5286 ┆ 17174   │


│ two-tower [test]                                      ┆ 0.205  ┆ 0.1227 ┆ 0.1827 ┆ 0.1803  ┆ 0.3509 ┆ 0.2689 ┆ 0.7129 ┆ 17174   │


│ two-tower + LambdaRank [test]                         ┆ 0.4622 ┆ 0.3182 ┆ 0.4615 ┆ 0.4557  ┆ 0.7633 ┆ 0.6435 ┆ 0.9245 ┆ 17174   │


└───────────────────────────────────────────────────────┴────────┴────────┴────────┴─────────┴────────┴────────┴────────┴─────────┘


## 6.2 Results

**Sample splits:** users are split 80/10/10 (train / validation / test); only users with at least one positive are
evaluated. Training-set numbers are on a 25% subsample of training users and use out-of-fold CTR and tower scores, so
they are comparable with validation/test (a large train–test gap would signal overfitting).

In [4]:
banner()
run.table("summary_test.csv", caption="Test users: every model on the full displayed slates")

,model,mrr,p1,ndcg10,gndcg10,hr10,r10,r50,n_users
0,Random [test],0.1005,0.0298,0.0850,0.0833,0.2266,0.1599,0.6081,17174
1,Popularity: users who interacted (all history) [test],0.3435,0.2012,0.3525,0.3479,0.6526,0.5403,0.8660,17174
2,"Popularity: clicks, last 7 days [test]",0.3311,0.1915,0.3386,0.3340,0.6349,0.5207,0.8403,17174
3,"Popularity: carts, last 90 days [test]",0.3344,0.1952,0.3424,0.3382,0.6353,0.5237,0.8459,17174
4,"Popularity: orders, last 365 days [test]",0.2929,0.1649,0.2954,0.2914,0.5646,0.4578,0.7942,17174
5,Impression CTR of item (training users) [test],0.2800,0.1705,0.2718,0.2681,0.5027,0.3985,0.6327,17174
6,Query-item CTR (training users) [test],0.1901,0.1073,0.1730,0.1701,0.3396,0.2579,0.5932,17174
7,"Query-item history engagement (clicks, 7 days) [test]",0.2441,0.1399,0.2343,0.2310,0.4461,0.3520,0.7084,17174
8,BM25 (query vs item text) [test],0.1096,0.0385,0.0959,0.0938,0.2352,0.1734,0.6327,17174
9,"User affinity: same category clicked, 7 days [test]",0.1106,0.0353,0.0969,0.0950,0.2508,0.1813,0.6500,17174


In [5]:
banner()
run.table("summary_all_splits.csv", caption="Training / validation / test performance")

,model,mrr,p1,ndcg10,gndcg10,hr10,r10,r50,n_users
0,Random [val],0.0993,0.0291,0.0826,0.0808,0.2239,0.1550,0.6060,17172
1,Popularity: users who interacted (all history) [val],0.3423,0.2007,0.3502,0.3460,0.6527,0.5375,0.8647,17172
2,"Popularity: clicks, last 7 days [val]",0.3297,0.1905,0.3370,0.3329,0.6337,0.5182,0.8365,17172
3,"Popularity: carts, last 90 days [val]",0.3321,0.1947,0.3379,0.3336,0.6321,0.5183,0.8420,17172
4,"Popularity: orders, last 365 days [val]",0.2890,0.1626,0.2892,0.2858,0.5585,0.4483,0.7880,17172
5,Impression CTR of item (training users) [val],0.2750,0.1674,0.2649,0.2611,0.4959,0.3899,0.6280,17172
6,Query-item CTR (training users) [val],0.1892,0.1073,0.1696,0.1671,0.3330,0.2500,0.5827,17172
7,"Query-item history engagement (clicks, 7 days) [val]",0.2435,0.1419,0.2308,0.2274,0.4390,0.3435,0.7018,17172
8,BM25 (query vs item text) [val],0.1083,0.0359,0.0947,0.0921,0.2389,0.1725,0.6243,17172
9,"User affinity: same category clicked, 7 days [val]",0.1110,0.0345,0.0975,0.0955,0.2570,0.1828,0.6466,17172


In [6]:
run.table("ranker_feature_importance.csv", highlight=()).data.head(25)

,feature,gain,split
0,ipop_click_7_pct,1.192250e+06,1229
1,ipop_click_7,6.146321e+05,425
2,ipop_order_365_pct,5.719744e+05,1072
3,u_wid_last_days,5.521573e+05,630
4,ipop_users,5.351455e+05,821
5,qpop_wid_click_7_pct,5.081073e+05,947
6,ctr_wid_pct,3.582690e+05,1182
7,u_shop_id_last_days,1.989971e+05,597
8,slate_share_shop_id,1.741417e+05,1615
9,ipop_cart_90,1.634927e+05,595
